# Cài đặt thư viện và xây dựng mô hình

In [ ]:
!pip install torch torchvision numpy pywavelets scikit-image opencv-python timm -q

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
import torch.nn.functional as F
from torchvision.models import convnext_tiny, convnext_base
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import os, random
from sklearn.metrics import accuracy_score


class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class ECAAttention(nn.Module):
    def __init__(self, channels, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(1, 2)
        y = self.conv(y)
        y = self.sigmoid(y)
        y = y.transpose(1, 2).unsqueeze(-1)
        return x * y.expand_as(x)

class ECABlock(nn.Module):
    def __init__(self, in_channels, out_channels, drop_prob=0.2):
        super().__init__()
        self.dws_conv = nn.Conv2d(in_channels, in_channels, kernel_size=7, stride=1, padding=3, groups=in_channels, bias=False)
        self.norm = nn.GroupNorm(1, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.gelu = nn.GELU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.eca = ECAAttention(out_channels)
        self.dropout = nn.Dropout2d(drop_prob)
        self.droppath = DropPath(drop_prob)
        self.proj = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.proj(x)
        out = self.dws_conv(x)
        out = self.norm(out)
        out = self.conv1(out)
        out = self.gelu(out)
        out = self.conv2(out)
        out = self.eca(out)
        out = self.dropout(out)
        out = self.droppath(out)
        return out + identity

class ConvNeXtWithECA(nn.Module):
    def __init__(self, num_classes=1000, drop_prob=0.2):
        super().__init__()
        base = convnext_tiny(weights="DEFAULT")  # pretrained
        self.features = nn.Sequential(*list(base.features.children()))
        self.eca = ECABlock(192, 192, drop_prob=drop_prob)
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 3:
                x = self.eca(x)
        x = self.pool(x)
        return torch.flatten(x, 1)

class TripletNet(nn.Module):
    def __init__(self, embeddingnet):
        super().__init__()
        self.embeddingnet = embeddingnet
    def forward(self, a, p, n):
        ea = self.embeddingnet(a)
        ep = self.embeddingnet(p)
        en = self.embeddingnet(n)
        return ea, ep, en


# Chuẩn bị dữ liệu

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, anchor_dir, pn_dir, transform=None):
        self.transform = transform
        self.anchor_data = {} 
        self.pn_data = {} 

        for cls in os.listdir(anchor_dir):
            cls_path = os.path.join(anchor_dir, cls)
            if os.path.isdir(cls_path):
                self.anchor_data[cls] = [os.path.join(cls_path, f) for f in os.listdir(cls_path)]

        for cls in os.listdir(pn_dir):
            cls_path = os.path.join(pn_dir, cls)
            if os.path.isdir(cls_path):
                self.pn_data[cls] = [os.path.join(cls_path, f) for f in os.listdir(cls_path)]

        self.classes = list(set(self.anchor_data.keys()) & set(self.pn_data.keys()))
        self.anchor_data = {cls: self.anchor_data[cls] for cls in self.classes}
        self.pn_data = {cls: self.pn_data[cls] for cls in self.classes}

    def __getitem__(self, index):
        anchor_class = self.classes[index % len(self.classes)]
        negative_class = random.choice([c for c in self.classes if c != anchor_class])

        anchor_path = random.choice(self.anchor_data[anchor_class]) 
        positive_path = random.choice(self.pn_data[anchor_class]) 
        negative_path = random.choice(self.pn_data[negative_class]) 

        a = Image.open(anchor_path).convert('RGB')
        p = Image.open(positive_path).convert('RGB')
        n = Image.open(negative_path).convert('RGB')

        if self.transform:
            a = self.transform(a)
            p = self.transform(p)
            n = self.transform(n)
        return a, p, n

    def __len__(self):
        return len(self.classes) * 50 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

anchor_path = "/kaggle/input/newgallery/gallery"
pn_path = "/kaggle/input/classify/CropCOnvData/train"     

dataset = TripletDataset(anchor_path, pn_path, transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)


embedding_net = ConvNeXtWithECA()
model = TripletNet(embedding_net).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.TripletMarginLoss(margin=0.7, p=2)

# Huấn luyện mô hình

In [ ]:
for epoch in range(70):
    model.train()
    total_loss = 0
    for a, p, n in loader:
        a, p, n = a.to(device), p.to(device), n.to(device)
        ea, ep, en = model(a, p, n)
        loss = criterion(ea, ep, en)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(loader):.4f}")

torch.save(model.state_dict(), "triplet_convnext_eca.pth")
print("✅ Mô hình đã lưu: triplet_convnext_eca.pth")

# Thực hiện đánh giá

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_model = ConvNeXtWithECA().to(device)
triplet_model = TripletNet(embedding_model).to(device)
triplet_model.load_state_dict(torch.load("triplet_convnext_eca.pth", map_location=device))
triplet_model.eval()

embedding_model = triplet_model.embeddingnet
embedding_model.to(device)
embedding_model.eval()

## Hàm trích xuất embedding

In [ ]:
def extract_embedding(img_path, transform, model):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model(img)
    return embedding.squeeze().cpu()

## Tạo thư viện embedding

In [ ]:
gallery_vectors = []
gallery_labels = []

for cls in os.listdir("/kaggle/input/newgallery/gallery"):
    cls_path = os.path.join("/kaggle/input/newgallery/gallery", cls)
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)
        vec = extract_embedding(img_path, transform, embedding_net)
        gallery_vectors.append(vec)
        gallery_labels.append(cls)

gallery_vectors = torch.stack(gallery_vectors)
torch.save({
    'vectors': gallery_vectors,
    'labels': gallery_labels
}, 'gallery_embeddings.pt')

## Đánh giá

In [ ]:
import os
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from PIL import Image

def extract_embedding(img_path, transform, model, device):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model(img)
    return embedding.squeeze().cpu()

def evaluate_accuracy_by_threshold(threshold, query_dir, gallery_vectors, gallery_labels, embedding_model, transform, device):
    correct = 0
    total = 0
    for cls in os.listdir(query_dir):
        cls_path = os.path.join(query_dir, cls)
        if not os.path.isdir(cls_path): continue
        for fname in os.listdir(cls_path):
            img_path = os.path.join(cls_path, fname)
            try:
                q_vec = extract_embedding(img_path, transform, embedding_model, device)
                sims = cosine_similarity(q_vec.unsqueeze(0), gallery_vectors)[0]
                match_found = any((lbl == cls and sim > threshold) for lbl, sim in zip(gallery_labels, sims))
                if match_found:
                    correct += 1
                total += 1
            except Exception:
                continue
    return correct / total if total > 0 else 0

def run_threshold_evaluation(query_dir, gallery_vectors, gallery_labels, embedding_model, transform, device):
    thresholds = np.linspace(0.1, 0.95, 18)
    accuracies = []

    for th in thresholds:
        acc = evaluate_accuracy_by_threshold(th, query_dir, gallery_vectors, gallery_labels,
                                              embedding_model, transform, device)
        print(f"Threshold {th:.2f} → Accuracy: {acc*100:.2f}%")
        accuracies.append(acc)

    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, np.array(accuracies)*100, marker='o', color='blue')
    plt.title("Accuracy theo Cosine Similarity Threshold")
    plt.xlabel("Cosine Similarity Threshold")
    plt.ylabel("Accuracy (%)")
    plt.grid(True)
    plt.xticks(np.round(thresholds, 2), rotation=45)
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

    return thresholds, accuracies


In [ ]:
thresholds, accuracies = run_threshold_evaluation(
    query_dir="/kaggle/input/classify/CropCOnvData/test",
    gallery_vectors=gallery_vectors,
    gallery_labels=gallery_labels,
    embedding_model=embedding_model,
    transform=transform,
    device=device
)


## Visualize không gian vector embedding

In [ ]:
pip install matplotlib scikit-learn umap-learn

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
import seaborn as sns

data = torch.load("gallery_embeddings.pt")
embeddings = data['vectors']  # shape (N, D)
labels = data['labels']

label_to_idx = {label: idx for idx, label in enumerate(sorted(set(labels)))}
numeric_labels = [label_to_idx[l] for l in labels]

use_umap = False

if use_umap:
    reducer = umap.UMAP(n_components=2, random_state=42)
else:
    reducer = TSNE(n_components=2, perplexity=30, random_state=42)

embeddings_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
palette = sns.color_palette("hsv", len(set(numeric_labels)))
sns.scatterplot(
    x=embeddings_2d[:, 0], y=embeddings_2d[:, 1],
    hue=numeric_labels,
    palette=palette,
    legend="full", alpha=0.8, s=40
)
plt.title("Embedding Visualization using " + ("UMAP" if use_umap else "t-SNE"))
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend(title="Class ID", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Kết quả truy xuất và thời gian xử lý

In [ ]:
import torch
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import os
from PIL import Image
import time 

data = torch.load("gallery_embeddings.pt")
gallery_vectors = data['vectors'] 
gallery_labels = data['labels']

embedding_model.eval()

query_dir = "/kaggle/input/classify/CropCOnvData/test"
top1_correct = 0
top5_correct = 0
total = 0

processing_times = []

for cls in tqdm(os.listdir(query_dir), desc="Classes"):
    cls_path = os.path.join(query_dir, cls)
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)

        start_time = time.time()

        query_vec = extract_embedding(img_path, transform, embedding_model, device)

        sims = cosine_similarity(query_vec.unsqueeze(0), gallery_vectors)[0]  
        topk_indices = sims.argsort()[-5:][::-1]

        topk_labels = [gallery_labels[i] for i in topk_indices]
        if cls == topk_labels[0]:
            top1_correct += 1
        if cls in topk_labels:
            top5_correct += 1
        total += 1

        elapsed_time = time.time() - start_time
        processing_times.append(elapsed_time)

print(f"Top-1 Accuracy: {top1_correct / total:.4f}")
print(f"Top-5 Accuracy: {top5_correct / total:.4f}")

avg_time = sum(processing_times) / len(processing_times)
print(f"Average processing time per image: {avg_time:.4f} seconds")
print(f"Max processing time: {max(processing_times):.4f} seconds")
print(f"Min processing time: {min(processing_times):.4f} seconds")
